# دستیار رزرو نوبت پزشک با Ollama و Tool Calling

این نوت‌بوک نسخهٔ ژوپیتر تمرین است و به API Key نیاز ندارد. Ollama باید روی همین کامپیوتر نصب و در حال اجرا باشد و مدل `qwen3:4b` دانلود شده باشد.

## پیش‌نیازها
در ترمینال سیستم اجرا کنید:
```bash
ollama pull qwen3:4b
pip install -U ollama
```
سپس سلول‌ها را از بالا به پایین اجرا کنید.

In [5]:
!pip install -U ollama


In [6]:
from ollama import chat

response = chat(
    model="qwen3:4b",
    messages=[
        {"role": "user", "content": "فقط بنویس: اتصال موفق است"}
    ]
)

print(response.message.content)

اتصال موفق است


In [7]:
# دستیار رزرو نوبت پزشک با Ollama و Tool Calling
# بدون کلید API؛ نیازمند نصب و اجرای Ollama روی همین کامپیوتر است.

import json
from pathlib import Path
from ollama import chat

MODEL = "qwen3:4b"
DATA_FILE = Path("appointments.json")

# برنامه ساختگی اولیه؛ برای تمرین قابل تغییر است.
INITIAL_SCHEDULE = {
    "شنبه": ["09:00", "10:00", "11:00"],
    "یکشنبه": ["09:00", "10:30"],
    "دوشنبه": ["14:00", "15:00", "16:00"],
}

def load_appointments():
    """نوبت‌های رزروشده را از فایل JSON می‌خواند؛ اگر فایل نباشد، فهرست خالی می‌سازد."""
    if DATA_FILE.exists():
        try:
            return json.loads(DATA_FILE.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            print("هشدار: فایل نوبت‌ها قابل خواندن نبود؛ از فهرست خالی استفاده می‌شود.")
    return []

appointments = load_appointments()

def get_available_slots(day: str = "") -> dict:
    """زمان‌های خالی پزشک را برمی‌گرداند. اگر day مشخص باشد، فقط همان روز را بررسی می‌کند."""
    if day and day not in INITIAL_SCHEDULE:
        return {"ok": False, "message": f"روز «{day}» در برنامه تعریف نشده است.",
                "available_slots": {}}

    days = [day] if day else list(INITIAL_SCHEDULE.keys())
    available = {}
    for current_day in days:
        booked_times = {
            item["time"] for item in appointments if item["day"] == current_day
        }
        free_times = [
            time for time in INITIAL_SCHEDULE[current_day]
            if time not in booked_times
        ]
        if free_times:
            available[current_day] = free_times

    return {
        "ok": True,
        "available_slots": available,
        "message": "زمان‌های خالی فعلی پزشک؛ فقط از این زمان‌ها برای رزرو استفاده کن."
    }

def book_appointment(patient_name: str, day: str, time: str,
                     reason: str = "چکاپ") -> dict:
    """برای بیمار در یک روز و ساعت مشخص نوبت ثبت می‌کند؛ نوبت پر یا نامعتبر را نمی‌پذیرد."""
    patient_name = patient_name.strip()
    if not patient_name:
        return {"ok": False, "message": "نام بیمار نمی‌تواند خالی باشد."}
    if day not in INITIAL_SCHEDULE:
        return {"ok": False, "message": f"روز «{day}» معتبر نیست."}
    if time not in INITIAL_SCHEDULE[day]:
        return {"ok": False, "message": f"ساعت {time} در برنامه {day} وجود ندارد."}

    if any(item["day"] == day and item["time"] == time for item in appointments):
        return {
            "ok": False,
            "message": f"زمان {day} ساعت {time} قبلاً رزرو شده است.",
            "available_slots": get_available_slots(day)["available_slots"]
        }

    appointment = {
        "patient_name": patient_name,
        "day": day,
        "time": time,
        "reason": reason.strip() or "چکاپ"
    }
    appointments.append(appointment)
    DATA_FILE.write_text(
        json.dumps(appointments, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    return {"ok": True, "appointment": appointment,
            "message": f"رزرو موفق: {patient_name}، {day} ساعت {time}."}

AVAILABLE_FUNCTIONS = {
    "get_available_slots": get_available_slots,
    "book_appointment": book_appointment,
}

SYSTEM_PROMPT = """
تو دستیار فارسی‌زبان رزرو نوبت یک کلینیک با یک پزشک هستی.
قوانین:
1) برای هر درخواست رزرو، ابتدا ابزار get_available_slots را صدا بزن.
2) اگر کاربر روز مشخصی گفته، ابزار را با همان روز فراخوانی کن و فقط از زمان‌های همان روز انتخاب کن.
3) اگر کاربر ساعت مشخصی خواسته، اگر آن ساعت در فهرست خالی همان روز/برنامه وجود دارد همان را رزرو کن؛ اگر روز مشخص نشده، روزی را انتخاب کن که آن ساعت در آن خالی است. اگر ساعت درخواستی پر یا ناموجود است، زمان‌های جایگزین را پیشنهاد بده و بدون تأیید کاربر جایگزین را رزرو نکن.
4) اگر کاربر روز و ساعت مشخص نکرده، رزرو را در اولین زمان خالی بر اساس ترتیب شنبه، یکشنبه، دوشنبه انجام بده.
5) فقط وقتی نام بیمار را از پیام‌ها می‌دانی، book_appointment را صدا بزن. اگر نام معلوم نیست، نام را از کاربر بپرس و رزرو نکن.
6) اگر همه زمان‌ها پر هستند، شفاف بگو نوبت خالی وجود ندارد و رزرو نکن.
7) فقط بعد از دریافت نتیجه موفق ابزار رزرو، تأیید نهایی شامل نام بیمار، روز، ساعت و علت مراجعه بده.
8) هرگز ادعا نکن رزرو انجام شده مگر ابزار نتیجه ok=true داده باشد. پاسخ‌ها کوتاه و فارسی باشند.
"""

def run_assistant(user_request: str, patient_name: str):
    """حلقه Tool Calling: پاسخ مدل، اجرای ابزار، و برگرداندن نتیجه ابزار به مدل."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"نام بیمار: {patient_name}\nدرخواست: {user_request}"}
    ]

    # محدودیت حلقه جلوی اجرای بی‌پایان را می‌گیرد.
    for _ in range(8):
        response = chat(
            model=MODEL,
            messages=messages,
            tools=[get_available_slots, book_appointment],
            options={"temperature": 0},
        )
        messages.append(response.message)
        tool_calls = response.message.tool_calls or []

        if not tool_calls:
            print("\nدستیار:", response.message.content or "پاسخی تولید نشد.")
            return

        for tool_call in tool_calls:
            name = tool_call.function.name
            args = tool_call.function.arguments or {}
            function = AVAILABLE_FUNCTIONS.get(name)

            if function is None:
                result = {"ok": False, "message": f"ابزار ناشناخته: {name}"}
            else:
                try:
                    result = function(**args)
                except TypeError as error:
                    result = {"ok": False, "message": f"آرگومان‌های ابزار نامعتبر است: {error}"}
                except Exception as error:
                    result = {"ok": False, "message": f"خطای اجرای ابزار: {error}"}

            print(f"\n[Tool Calling] {name}({args})")
            print("[نتیجه ابزار]", json.dumps(result, ensure_ascii=False))
            messages.append({
                "role": "tool",
                "tool_name": name,
                "content": json.dumps(result, ensure_ascii=False)
            })

    print("حلقه پس از ۸ مرحله متوقف شد؛ لطفاً درخواست را ساده‌تر بیان کن.")

def run_tests():
    """چهار سناریوی تمرین را با داده‌های جداگانه آزمایش می‌کند."""
    global appointments
    original = appointments[:]
    try:
        appointments = []
        print("\n===== تست ۱: اولین زمان خالی =====")
        run_assistant("برای چکاپ نوبت می‌خواهم.", "آزمایش اول")

        appointments = []
        print("\n===== تست ۲: فقط دوشنبه =====")
        run_assistant("نوبت روز دوشنبه می‌خواهم.", "آزمایش دوم")

        appointments = [{"patient_name": "بیمار قبلی", "day": "شنبه",
                         "time": "10:00", "reason": "چکاپ"}]
        print("\n===== تست ۳: ساعت ۱۰ صبح که قبلاً رزرو شده =====")
        run_assistant("نوبت ساعت 10 صبح می‌خواهم.", "آزمایش سوم")

        appointments = [
            {"patient_name": "بیمار", "day": day, "time": time, "reason": "چکاپ"}
            for day, times in INITIAL_SCHEDULE.items() for time in times
        ]
        print("\n===== تست ۴: همه نوبت‌ها پر =====")
        run_assistant("برای چکاپ نوبت می‌خواهم.", "آزمایش چهارم")
    finally:
        appointments = original
        DATA_FILE.write_text(json.dumps(appointments, ensure_ascii=False, indent=2),
                             encoding="utf-8")
        print("\nتست‌ها تمام شد؛ فهرست واقعی نوبت‌ها بازیابی شد.")

def main():
    print("دستیار رزرو نوبت پزشک با Ollama (بدون API Key)")
    print("برای خروج، exit و برای اجرای چهار تست، test بنویس.\n")
    while True:
        user_request = input("درخواست بیمار: ").strip()
        if user_request.lower() in {"exit", "quit", "خروج"}:
            break
        if user_request.lower() == "test":
            run_tests()
            continue
        if not user_request:
            continue
        patient_name = input("نام بیمار: ").strip()
        if not patient_name:
            print("نام بیمار الزامی است؛ رزرو انجام نشد.\n")
            continue
        try:
            run_assistant(user_request, patient_name)
        except Exception as error:
            print("\nخطا در اتصال به Ollama یا مدل:")
            print(error)
            print(f"بررسی کن Ollama اجرا باشد و مدل {MODEL} نصب شده باشد.")
        print()



## اجرای دستیار
در سلول بعدی درخواست و نام بیمار را وارد کنید. مثال: «برای چکاپ نوبت می‌خواهم».

In [8]:
request = input('درخواست بیمار: ').strip()
patient_name = input('نام بیمار: ').strip()
if request and patient_name:
    run_assistant(request, patient_name)
else:
    print('درخواست و نام بیمار را وارد کنید.')


درخواست بیمار:  رزرو نوبت 
نام بیمار:  حسین



[Tool Calling] get_available_slots({'day': None})
[نتیجه ابزار] {"ok": true, "available_slots": {"شنبه": ["09:00", "10:00", "11:00"], "یکشنبه": ["09:00", "10:30"], "دوشنبه": ["14:00", "15:00", "16:00"]}, "message": "زمان‌های خالی فعلی پزشک؛ فقط از این زمان‌ها برای رزرو استفاده کن."}

دستیار: لطفاً علت مراجعه خود را ذکر کن تا نوبت را ثبت کنیم.


## اجرای چهار تست تمرین
این تست‌ها از داده‌های ساختگی استفاده می‌کنند و در پایان نوبت‌های ذخیره‌شدهٔ قبلی را بازیابی می‌کنند.

In [ ]:
run_tests()



===== تست ۱: اولین زمان خالی =====

[Tool Calling] get_available_slots({'day': None})
[نتیجه ابزار] {"ok": true, "available_slots": {"شنبه": ["09:00", "10:00", "11:00"], "یکشنبه": ["09:00", "10:30"], "دوشنبه": ["14:00", "15:00", "16:00"]}, "message": "زمان‌های خالی فعلی پزشک؛ فقط از این زمان‌ها برای رزرو استفاده کن."}

[Tool Calling] book_appointment({'patient_name': 'آزمایش اول', 'day': 'شنبه', 'time': '09:00', 'reason': 'چکاپ'})
[نتیجه ابزار] {"ok": true, "appointment": {"patient_name": "آزمایش اول", "day": "شنبه", "time": "09:00", "reason": "چکاپ"}, "message": "رزرو موفق: آزمایش اول، شنبه ساعت 09:00."}

دستیار: رزرو موفق: آزمایش اول، شنبه ساعت 09:00 برای چکاپ.

===== تست ۲: فقط دوشنبه =====

[Tool Calling] get_available_slots({'day': 'دوشنبه'})
[نتیجه ابزار] {"ok": true, "available_slots": {"دوشنبه": ["14:00", "15:00", "16:00"]}, "message": "زمان‌های خالی فعلی پزشک؛ فقط از این زمان‌ها برای رزرو استفاده کن."}

دستیار: لطفاً علت مراجعه خود را بگویید تا نوبت روز دوشنبه ساعت 14:00 را ث